# The quantum metric: curved surfaces, quantum distances and graphene

**Professor Nano Labs · Quantum geometry**

**Try it live in your browser:** [The quantum metric lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/quantum-metric/)
**Watch first:** [Quantum Geometry; Differential geometry meets Bloch states](https://www.youtube.com/watch?v=FrfRj9YWfaE)
**Watch later:** [Deriving Berry phase, Berry connection & Berry curvature](https://www.youtube.com/watch?v=6xctcmwR8wo) · [Mapping the qubit state onto the Bloch Sphere](https://www.youtube.com/watch?v=lqWSziZJsLs) · [Deriving the Bloch's theorem](https://www.youtube.com/watch?v=Bj2rNT5fa9s)

Geometry starts with distance. On a surface $\mathbf r(u^1,u^2)$, the metric $g_{ij}=\mathbf e_i\cdot\mathbf e_j$, with $\mathbf e_i=\partial\mathbf r/\partial u^i$, turns coordinate steps into lengths, $ds^2=g_{ij}du^idu^j$; and the Gaussian curvature $K$, which depends on the metric alone, sets how much a vector carried round a loop turns. For quantum states, the distance between neighbours,
$$1-|\langle u_{\mathbf k}|u_{\mathbf k+d\mathbf k}\rangle|^2\approx g_{ij}\,dk_i\,dk_j,\qquad Q_{ij}=\langle\partial_iu|(1-|u\rangle\langle u|)|\partial_ju\rangle=g_{ij}-\tfrac i2\Omega_{ij},$$
defines the **quantum metric** $g_{ij}$, the real part of the quantum geometric tensor, whose imaginary part is the Berry curvature.

The notebook checks every number quoted in the lab's "Predict first" answers. It needs NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PI = np.pi
wrap = lambda a: (np.asarray(a) + PI) % (2 * PI) - PI

# the four surfaces, in their chart coordinates (u1, u2)
SURF = {
    'plane':    lambda a, b: np.array([a, b, 0.0]),
    'cylinder': lambda a, b: np.array([np.cos(a), np.sin(a), b]),                                  # angle around, height
    'sphere':   lambda a, b: np.array([np.cos(b) * np.cos(a), np.cos(b) * np.sin(a), np.sin(b)]),   # longitude, latitude
    'saddle':   lambda a, b: np.array([a, b, (a * a - b * b) / 2]),
}

def frame(r, u, h=1e-5):
    a, b = u
    e1 = (r(a + h, b) - r(a - h, b)) / (2 * h); e2 = (r(a, b + h) - r(a, b - h)) / (2 * h)
    n = np.cross(e1, e2); n /= np.linalg.norm(n)
    g = np.array([[e1 @ e1, e1 @ e2], [e1 @ e2, e2 @ e2]])
    return e1, e2, n, g

def gauss_K(r, u, h=1e-3):
    a, b = u
    _, _, n, g = frame(r, u)
    r11 = (r(a + h, b) - 2 * r(a, b) + r(a - h, b)) / h**2
    r22 = (r(a, b + h) - 2 * r(a, b) + r(a, b - h)) / h**2
    r12 = (r(a + h, b + h) - r(a + h, b - h) - r(a - h, b + h) + r(a - h, b - h)) / (4 * h * h)
    L = np.array([[r11 @ n, r12 @ n], [r12 @ n, r22 @ n]])
    return np.linalg.det(L) / np.linalg.det(g)

for name, u, exact in [('plane', (0.3, -0.7), 0), ('cylinder', (1.0, 0.4), 0), ('sphere', (0.5, 0.8), 1), ('saddle', (0.4, -0.6), -1 / (1 + 0.16 + 0.36)**2)]:
    K = gauss_K(SURF[name], u)
    print(f'{name:9s} K = {K:+.6f}   exact {exact:+.6f}   metric {np.round(frame(SURF[name], u)[3], 4).tolist()}')
    assert abs(K - exact) < 1e-5

## 1 · Carrying an arrow round a loop

The loop is the **metric ellipse** $g_{ij}du^idu^j=\varepsilon^2$ around a point: to first order a circle of radius $\varepsilon$ on the surface. An arrow is carried round it by **parallel transport**; here we use the simplest version, projecting the arrow onto each new tangent plane and restoring its length, which becomes exact as the steps shrink. Its turn should equal the curvature inside the loop, $\iint K\,dA$ (Gauss–Bonnet), and $K\times\pi\varepsilon^2$ for a small loop.

In [ ]:
def inv_sqrt(g):
    w, v = np.linalg.eigh(g)
    return v @ np.diag(w**-0.5) @ v.T

def loop(r, u, eps, N):
    W = inv_sqrt(frame(r, u)[3]); t = 2 * PI * np.arange(N) / N
    return np.array(u)[None, :] + eps * (np.stack([np.cos(t), np.sin(t)], 1) @ W.T)

def transport_turn(r, u, eps, N=20000):
    pts = loop(r, u, eps, N)
    e1, _, n0, _ = frame(r, pts[0]); v0 = e1 / np.linalg.norm(e1); v = v0.copy()
    for p in list(pts[1:]) + [pts[0]]:
        n = frame(r, p)[2]
        v = v - (v @ n) * n; v /= np.linalg.norm(v)
    return np.arctan2(n0 @ np.cross(v0, v), v0 @ v)

def curvature_inside(r, u, eps, nr=40, na=200):
    W = inv_sqrt(frame(r, u)[3]); total = 0.0
    s = (np.arange(nr) + 0.5) / nr; t = 2 * PI * np.arange(na) / na
    for si in s:
        for ti in t:
            q = np.array(u) + eps * si * (W @ np.array([np.cos(ti), np.sin(ti)]))
            g = frame(r, q)[3]
            total += gauss_K(r, q) * np.sqrt(np.linalg.det(g)) * eps**2 * si * np.linalg.det(W) / nr * 2 * PI / na
    return total

# Question 1: a cylinder does not turn the arrow
turn = transport_turn(SURF['cylinder'], (0.5, 0.2), 0.5)
print(f'cylinder: turn {np.degrees(turn):.4f} degrees'); assert abs(turn) < 1e-6

# Question 2: a loop of size 0.5 at the centre of the sphere's chart, and of the saddle
for name in ['sphere', 'saddle']:
    T, I = transport_turn(SURF[name], (0, 0), 0.5), curvature_inside(SURF[name], (0, 0), 0.5)
    print(f'{name}: turn {np.degrees(T):+.2f} degrees, curvature inside {np.degrees(I):+.2f}, K x area {np.degrees(gauss_K(SURF[name], (0, 0)) * PI * 0.25):+.2f}')
    assert abs(T - I) < 2e-3
    assert round(np.degrees(T)) == (44 if name == 'sphere' else -38)
assert round(np.degrees(PI * 0.25)) == 45

# turn against loop size on the sphere: K x area for small loops
for eps in [0.1, 0.2, 0.4]:
    T = transport_turn(SURF['sphere'], (0, 0), eps)
    print(f'eps = {eps}: turn / (K pi eps^2) = {T / (PI * eps**2):.4f}')

### Question 3 · The world map

At latitude $\lambda$ a degree of longitude covers $\cos\lambda$ of the distance of a degree of latitude: $g_{\varphi\varphi}=\cos^2\lambda$, $g_{\lambda\lambda}=1$. The ellipse of equal distances is wider than it is tall by $1/\cos\lambda$.

In [ ]:
g = frame(SURF['sphere'], (0.3, PI / 3))[3]
W = inv_sqrt(g)
ratio = W[0, 0] / W[1, 1]
print('metric at latitude 60 degrees:', np.round(g, 6).tolist(), ' width / height of the ellipse:', round(ratio, 6))
assert abs(g[0, 0] - 0.25) < 1e-8 and abs(ratio - 2) < 1e-6

# the metric ellipses of the sphere's chart
fig, ax = plt.subplots(figsize=(7, 3.2))
for a in np.linspace(-2.8, 2.8, 8):
    for b in np.linspace(-1.15, 1.15, 6):
        Wq = inv_sqrt(frame(SURF['sphere'], (a, b))[3]); t = np.linspace(0, 2 * PI, 60)
        e = 0.13 * (Wq @ np.stack([np.cos(t), np.sin(t)]))
        ax.plot(a + e[0], b + e[1], color='#946600')
ax.set_aspect('equal'); ax.set_xlabel('longitude'); ax.set_ylabel('latitude'); ax.set_title('equal distances on the sphere\'s chart')
plt.tight_layout(); plt.show()

## 2 · The distance between quantum states (Questions 4 to 6)

For a spin-½, $|{+}\mathbf n\rangle=(\cos\frac\theta2,\ e^{i\varphi}\sin\frac\theta2)$. The quantum distance from a reference state is $1-|\langle\psi_0|\psi\rangle|^2$; the length of a path is the sum of $\arccos|\langle\psi_j|\psi_{j+1}\rangle|$ over small steps, which tends to $\int ds$ with $ds^2=\tfrac14(d\theta^2+\sin^2\theta\,d\varphi^2)$: a sphere of radius ½.

In [ ]:
def up(th, ph):
    return np.array([np.cos(th / 2), np.exp(1j * ph) * np.sin(th / 2)])

def loop_states(th, N=4000, scramble=None):
    st = [up(th, 2 * PI * j / N) for j in range(N + 1)]
    if scramble is not None:
        st = [s * np.exp(2j * PI * scramble.random()) if 0 < j < N else s for j, s in enumerate(st)]
    return st

def odometer(st):
    return np.concatenate([[0], np.cumsum([np.arccos(min(1, abs(np.vdot(a, b)))) for a, b in zip(st[:-1], st[1:])])])

# Question 4: the equator
st = loop_states(PI / 2); od = odometer(st); N = len(st) - 1
d_half = 1 - abs(np.vdot(st[0], st[N // 2]))**2
print(f'equator: half way, 1 - |<psi0|psi>|^2 = {d_half:.6f}; travelled {od[N // 2]:.6f} (pi/2 = {PI / 2:.6f}); whole loop {od[-1]:.6f}')
assert abs(d_half - 1) < 1e-12 and abs(od[N // 2] - PI / 2) < 1e-6 and abs(od[-1] - PI) < 1e-6
assert abs(1 - abs(np.vdot(st[0], st[-1]))**2) < 1e-12

# Question 5: scrambled phases change nothing measurable
rng = np.random.default_rng(3)
st2 = loop_states(PI / 3, scramble=rng); st1 = loop_states(PI / 3)
d1 = [1 - abs(np.vdot(st1[0], s))**2 for s in st1]; d2 = [1 - abs(np.vdot(st2[0], s))**2 for s in st2]
print('largest change in the distance from the start:', np.max(np.abs(np.array(d1) - d2)), ' in the length:', abs(odometer(st1)[-1] - odometer(st2)[-1]))
assert np.max(np.abs(np.array(d1) - d2)) < 1e-12 and abs(odometer(st1)[-1] - odometer(st2)[-1]) < 1e-9

# Question 6: the 60 and 120 degree loops
for deg, berry in [(60, -0.5), (120, -1.5)]:
    th = np.radians(deg); L = odometer(loop_states(th))[-1]
    print(f'{deg} degrees: length {L:.4f} (pi sin = {PI * np.sin(th):.4f}), Berry phase {-PI * (1 - np.cos(th)) / PI:+.2f} pi')
    assert round(L, 2) == 2.72 and abs(-(1 - np.cos(th)) - berry) < 1e-12

ths = np.linspace(0, PI, 181)
plt.figure(figsize=(6, 3.4))
plt.plot(np.degrees(ths), PI * np.sin(ths), 'k', label='length of the loop')
plt.plot(np.degrees(ths), PI * (1 - np.cos(ths)), 'r--', label='|Berry phase|')
plt.xlabel('cone angle (degrees)'); plt.legend(); plt.tight_layout(); plt.show()

## 3 · The quantum metric of graphene (Questions 7 and 8)

Graphene as in card 1 (carbon–carbon distance 1, $t=1$, sublattice mass $M$, Haldane's $t_2$). Here the metric is computed straight from the definition, from overlaps of eigenvectors of the Bloch Hamiltonian on a small cross of points, and compared with the two-band formula $g_{ij}=\tfrac14\partial_i\mathbf n\cdot\partial_j\mathbf n$ and with $\sqrt{\det g}=|\Omega|/2$.

In [ ]:
S3 = np.sqrt(3)
DEL = np.array([[0, 1], [-S3 / 2, -0.5], [S3 / 2, -0.5]])
BB = np.array([DEL[1] - DEL[2], DEL[2] - DEL[0], DEL[0] - DEL[1]])
K = np.array([4 * PI / (3 * S3), 0.0])
sx = np.array([[0, 1], [1, 0]], complex); sy = np.array([[0, -1j], [1j, 0]]); sz = np.diag([1.0 + 0j, -1.0])

def dvec(k, M, t2=0.0, phi=PI / 2):
    q, p = DEL @ k, BB @ k
    return np.array([np.cos(q).sum(), np.sin(q).sum(), M - 2 * t2 * np.sin(phi) * np.sin(p).sum()])

def lower(k, *a):
    d = dvec(k, *a)
    return np.linalg.eigh(d[0] * sx + d[1] * sy + d[2] * sz)[1][:, 0]

def metric_fid(k, *a, h=1e-4):
    # g_ij from 1 - |<u(k)|u(k + dk)>|^2 along x, y and the diagonal
    u0 = lower(k, *a)
    # symmetric steps cancel the third-order term, so the error is of order h^2
    f = lambda dk: (2 - abs(np.vdot(u0, lower(k + dk, *a)))**2 - abs(np.vdot(u0, lower(k - dk, *a)))**2) / 2
    gxx, gyy = f(np.array([h, 0])) / h**2, f(np.array([0, h])) / h**2
    gxy = (f(np.array([h, h])) / h**2 - gxx - gyy) / 2
    return np.array([[gxx, gxy], [gxy, gyy]])

def metric_n(k, *a, h=1e-6):
    n = lambda q: dvec(q, *a) / np.linalg.norm(dvec(q, *a))
    nx = (n(k + [h, 0]) - n(k - [h, 0])) / (2 * h); ny = (n(k + [0, h]) - n(k - [0, h])) / (2 * h)
    g = np.array([[nx @ nx, nx @ ny], [nx @ ny, ny @ ny]]) / 4
    return g, 0.5 * n(k) @ np.cross(nx, ny)

rng = np.random.default_rng(0)
for _ in range(5):
    k = rng.uniform(-3, 3, 2); args = (rng.uniform(-0.4, 0.4), rng.uniform(0, 0.3), rng.uniform(-PI, PI))
    gf, (gn, om) = metric_fid(k, *args), metric_n(k, *args)
    print(f'k = {np.round(k, 2)}: g from overlaps {np.round(gf, 4).ravel()}, from n {np.round(gn, 4).ravel()};  sqrt(det g) = {np.sqrt(np.linalg.det(gn)):.6f}, |Omega|/2 = {abs(om) / 2:.6f}')
    assert np.allclose(gf, gn, rtol=1e-5, atol=1e-7) and abs(np.sqrt(np.linalg.det(gn)) - abs(om) / 2) < 1e-6

# Question 7: halve the mass, four times the metric at K
for M, quoted in [(0.2, 28.1), (0.1, 112.5)]:
    trK = np.trace(metric_n(K, M)[0])
    print(f'M = {M}: tr g at K = {trK:.4f}   v^2 / 2M^2 = {2.25 / (2 * M * M):.4f}')
    assert abs(trK - 2.25 / (2 * M * M)) < 1e-3 * trK and round(trK, 1) == quoted

# Question 8: plain graphene, the path straight across K: the odometer jumps by pi/2
ys = np.linspace(-1.2, 1.2, 1201)
for M in [0.0, 0.2]:
    st = [lower(np.array([K[0], y]), M) for y in ys]
    od = np.concatenate([[0], np.cumsum([np.arccos(min(1, abs(np.vdot(a, b)))) for a, b in zip(st[:-1], st[1:])])])
    jump = od[601] - od[599]
    print(f'M = {M}: distance across K {od[-1]:.4f}, of which the two steps at K {jump:.4f}')
    if M == 0: assert abs(jump - PI / 2) < 2e-3          # pi/2, plus the ordinary change over two steps of 0.002
    else: assert jump < 0.05                             # with a gap, no jump: just fast, smooth change

# the metric over the zone, for graphene on a substrate
g = np.linspace(-3, 3, 161); KX, KY = np.meshgrid(g, g)
tr = np.array([[np.trace(metric_n(np.array([x, y]), 0.2)[0]) for x in g] for y in g])
plt.figure(figsize=(5.2, 4.4))
plt.pcolormesh(KX, KY, np.tanh(tr / 10), cmap='YlOrBr', shading='auto')
hexagon = np.array([[4 * PI / (3 * S3) * np.cos(a), 4 * PI / (3 * S3) * np.sin(a)] for a in np.arange(7) * PI / 3])
plt.plot(hexagon[:, 0], hexagon[:, 1], 'k--'); plt.gca().set_aspect('equal')
plt.title('tr g, graphene on a substrate (M = 0.2)'); plt.xlabel('$k_x$'); plt.ylabel('$k_y$'); plt.tight_layout(); plt.show()

## Things to try

1. **A torus.** Add the surface $\mathbf r=((2+\cos v)\cos u,(2+\cos v)\sin u,\sin v)$. Where is its curvature positive, where negative, and what is $\iint K\,dA$ over the whole torus?
2. **Geodesic circles.** Replace the metric ellipse by a true geodesic circle and check that the turn is exactly $2\pi(1-\cos\varepsilon)$ on the unit sphere.
3. **The Qi–Wu–Zhang model.** Map its quantum metric for $m=1$ and $m=3$, and check that $\frac1{2\pi}\int\mathrm{tr}\,g\ \ge\ |C|$.
4. **A three-band model.** For more than two bands $\sqrt{\det g}=|\Omega|/2$ no longer holds, only $\mathrm{tr}\,g\ge|\Omega|$. Find an example.